In [1]:
# ============================================================
# NEURAL OBJECT RECONSTRUCTION & RELIGHTING LAB
# RAW OBJECT DATASET PIPELINE
#
# Objaverse → filtering → 200 objects → Hugging Face
# ============================================================

!pip -q install -U \
    objaverse \
    huggingface_hub \
    pandas \
    tqdm

In [2]:
# ============================================================
# IMPORTS AND CONFIGURATION
# ============================================================

import os
import gc
import json
import time
import shutil
import random
from pathlib import Path
from collections import Counter

import pandas as pd
import objaverse

from tqdm.auto import tqdm

from huggingface_hub import (
    HfApi,
    login,
    create_repo,
    delete_repo,
    snapshot_download
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
# ============================================================
# PROJECT CONFIGURATION
# ============================================================

HF_USERNAME = "ndeda"

HF_DATASET = (
    f"{HF_USERNAME}/"
    "neural-object-reconstruction-raw"
)

# Exactly how many objects we want
TOTAL_OBJECTS = 200

# Objects per local upload batch
BATCH_SIZE = 25

# Number of objects per semantic category
OBJECTS_PER_CATEGORY = 20

# Reproducible selection
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

# Local project directory
PROJECT_DIR = Path(
    "/content/neural_object_reconstruction"
)

# IMPORTANT:
# This is separate from Objaverse's own cache.
OBJAVERSE_CACHE = Path(
    "/root/.objaverse"
)

# Our metadata directory
METADATA_DIR = (
    PROJECT_DIR / "metadata"
)

# Temporary batches
BATCHES_DIR = (
    PROJECT_DIR / "batches"
)

# Create directories
METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BATCHES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("HF dataset:")
print(HF_DATASET)

print("\nProject directory:")
print(PROJECT_DIR)

print("\nObjaverse cache:")
print(OBJAVERSE_CACHE)

HF dataset:
ndeda/neural-object-reconstruction-raw

Project directory:
/content/neural_object_reconstruction

Objaverse cache:
/root/.objaverse


In [4]:
# ============================================================
# HUGGING FACE AUTHENTICATION
# ============================================================

print(
    "Logging into Hugging Face..."
)

login()

api = HfApi()

whoami = api.whoami()

print("\nAuthenticated as:")
print(whoami["name"])

Logging into Hugging Face...

Authenticated as:
ndeda


In [5]:
# ============================================================
# DELETE OLD DATASET REPOSITORY
# ============================================================

print("=" * 70)
print("WARNING")
print("=" * 70)

print(
    f"This will permanently delete:\n"
    f"{HF_DATASET}"
)

confirmation = input(
    "\nType DELETE to continue: "
)

if confirmation != "DELETE":
    raise RuntimeError(
        "Deletion cancelled."
    )

print(
    "\nDeleting existing repository..."
)

delete_repo(
    repo_id=HF_DATASET,
    repo_type="dataset",
    missing_ok=True
)

print(
    "✓ Old repository deleted."
)

WARNING
This will permanently delete:
ndeda/neural-object-reconstruction-raw

Type DELETE to continue: DELETE

Deleting existing repository...
✓ Old repository deleted.


In [6]:
# ============================================================
# CREATE PUBLIC DATASET REPOSITORY
# ============================================================

print(
    "Creating PUBLIC dataset repository..."
)

create_repo(
    repo_id=HF_DATASET,
    repo_type="dataset",
    private=False,
    exist_ok=False
)

print(
    "\n✓ PUBLIC repository created:"
)

print(
    f"https://huggingface.co/datasets/{HF_DATASET}"
)

Creating PUBLIC dataset repository...

✓ PUBLIC repository created:
https://huggingface.co/datasets/ndeda/neural-object-reconstruction-raw


In [7]:
# ============================================================
# CLEAN OUR PROJECT DIRECTORY
# ============================================================

if PROJECT_DIR.exists():

    print(
        f"Deleting previous project directory:\n"
        f"{PROJECT_DIR}"
    )

    shutil.rmtree(
        PROJECT_DIR
    )

# Recreate
METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BATCHES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "✓ Local project reset."
)

print(
    "\nObjaverse cache was NOT deleted:"
)

print(
    OBJAVERSE_CACHE
)

Deleting previous project directory:
/content/neural_object_reconstruction
✓ Local project reset.

Objaverse cache was NOT deleted:
/root/.objaverse


In [8]:
# ============================================================
# LOAD OBJAVERSE ANNOTATIONS
# ============================================================

print(
    "Loading Objaverse annotations..."
)

annotations = objaverse.load_annotations()

print(
    "\nTotal objects:",
    len(annotations)
)

Loading Objaverse annotations...


100%|██████████| 160/160 [01:03<00:00,  2.52it/s]


Total objects: 798759


In [9]:
# ============================================================
# EXCLUSION RULES
# ============================================================

EXCLUDED_CATEGORIES = {
    "weapons-military",
    "weapons",
    "weapon",
    "military",
    "firearms",
    "guns",
    "adult",
    "nsfw",
    "anatomy",
    "medical",
    "pornography",
}

EXCLUDED_KEYWORDS = {
    "gun",
    "rifle",
    "pistol",
    "firearm",
    "weapon",
    "sword",
    "knife",
    "dagger",
    "grenade",
    "missile",
    "bullet",
    "ammunition",
    "porn",
    "nsfw",
}

In [10]:
# ============================================================
# TARGET CATEGORIES
# ============================================================

TARGET_CATEGORIES = {

    "furniture": {
        "chair",
        "table",
        "desk",
        "sofa",
        "couch",
        "cabinet",
        "shelf",
        "stool",
        "bench",
        "bed",
        "wardrobe",
        "drawer",
        "bookcase",
        "dresser"
    },

    "kitchen": {
        "mug",
        "cup",
        "plate",
        "bowl",
        "glass",
        "bottle",
        "pot",
        "pan",
        "kettle",
        "toaster",
        "blender",
        "fork",
        "spoon",
        "container",
        "pitcher",
        "teapot"
    },

    "electronics": {
        "phone",
        "smartphone",
        "laptop",
        "tablet",
        "computer",
        "keyboard",
        "mouse",
        "camera",
        "headphone",
        "headphones",
        "speaker",
        "monitor",
        "controller",
        "console",
        "microphone",
        "router",
        "watch"
    },

    "tools": {
        "hammer",
        "screwdriver",
        "drill",
        "wrench",
        "pliers",
        "saw",
        "tool",
        "clamp",
        "cutter",
        "chisel",
        "flashlight",
        "stapler"
    },

    "lighting": {
        "lamp",
        "lantern",
        "light",
        "lightbulb",
        "lampshade",
        "torch",
        "chandelier"
    },

    "clothing": {
        "shoe",
        "sneaker",
        "boot",
        "sandal",
        "hat",
        "cap",
        "backpack",
        "bag",
        "wallet",
        "belt",
        "glasses",
        "jacket",
        "shirt",
        "pants",
        "scarf",
        "helmet",
        "suitcase"
    },

    "toys": {
        "toy",
        "doll",
        "robot",
        "teddy",
        "puzzle",
        "dice",
        "chess",
        "rubik",
        "figurine",
        "action figure",
        "building block"
    },

    "musical": {
        "guitar",
        "violin",
        "cello",
        "piano",
        "drum",
        "cymbal",
        "flute",
        "trumpet",
        "saxophone",
        "clarinet",
        "harmonica",
        "tambourine",
        "ukulele",
        "banjo"
    },

    "sports": {
        "football",
        "soccer",
        "basketball",
        "tennis",
        "racket",
        "baseball",
        "golf",
        "skateboard",
        "bicycle",
        "ball",
        "club",
        "sports",
        "camping"
    },

    "decorative": {
        "vase",
        "statue",
        "sculpture",
        "ornament",
        "decoration",
        "clock",
        "mirror",
        "frame",
        "candle",
        "box",
        "tray"
    }
}

print(
    "Categories:",
    len(TARGET_CATEGORIES)
)

for category in TARGET_CATEGORIES:
    print(
        f"  {category}: "
        f"{OBJECTS_PER_CATEGORY} objects"
    )

print(
    "\nTotal target:",
    len(TARGET_CATEGORIES)
    * OBJECTS_PER_CATEGORY
)

Categories: 10
  furniture: 20 objects
  kitchen: 20 objects
  electronics: 20 objects
  tools: 20 objects
  lighting: 20 objects
  clothing: 20 objects
  toys: 20 objects
  musical: 20 objects
  sports: 20 objects
  decorative: 20 objects

Total target: 200


In [11]:
# ============================================================
# METADATA EXTRACTION
# ============================================================

def extract_metadata(
    uid,
    metadata
):

    name = str(
        metadata.get(
            "name",
            ""
        )
    ).lower().strip()

    categories = [
        str(
            x.get("name", "")
        ).lower().strip()

        for x in metadata.get(
            "categories",
            []
        )

        if isinstance(x, dict)
    ]

    tags = [
        str(
            x.get("name", "")
        ).lower().strip()

        for x in metadata.get(
            "tags",
            []
        )

        if isinstance(x, dict)
    ]

    license_raw = str(
        metadata.get(
            "license",
            ""
        )
    ).lower().strip()

    return {
        "uid": uid,
        "name": name,
        "categories": categories,
        "tags": tags,
        "license_raw": license_raw,
        "is_downloadable": bool(
            metadata.get(
                "isDownloadable",
                False
            )
        ),
        "is_age_restricted": bool(
            metadata.get(
                "isAgeRestricted",
                False
            )
        ),
        "face_count": metadata.get(
            "faceCount",
            0
        ),
        "vertex_count": metadata.get(
            "vertexCount",
            0
        ),
        "viewer_url": metadata.get(
            "viewerUrl",
            ""
        ),
        "source_uri": metadata.get(
            "uri",
            ""
        ),
        "creator": (
            metadata.get(
                "user",
                {}
            ) or {}
        ).get(
            "username",
            ""
        )
    }

In [12]:
# ============================================================
# LICENSE HANDLING
# ============================================================

def normalize_license(
    license_raw
):

    raw = str(
        license_raw
    ).lower().strip()

    mapping = {
        "by": "CC-BY",
        "cc-by": "CC-BY",
        "cc by": "CC-BY",
        "cc0": "CC0",
        "zero": "CC0",
    }

    return mapping.get(
        raw,
        raw.upper()
    )


ALLOWED_LICENSES = {
    "CC0",
    "CC-BY"
}

In [13]:
# ============================================================
# OBJECT CLASSIFICATION
# ============================================================

def classify_object(obj):

    text = " ".join(
        [
            obj["name"],
            *obj["tags"]
        ]
    ).lower()

    scores = {}

    for category, keywords in (
        TARGET_CATEGORIES.items()
    ):

        score = 0

        for keyword in keywords:

            if keyword in text:
                score += 1

        if score > 0:
            scores[category] = score

    if not scores:
        return None

    return max(
        scores,
        key=scores.get
    )

In [14]:
# ============================================================
# OBJECT FILTERING
# ============================================================

MIN_FACES = 500
MAX_FACES = 500_000


def is_excluded(obj):

    # --------------------------------------------------------
    # License
    # --------------------------------------------------------

    normalized_license = normalize_license(
        obj["license_raw"]
    )

    if normalized_license not in ALLOWED_LICENSES:
        return True

    # --------------------------------------------------------
    # Downloadability
    # --------------------------------------------------------

    if not obj["is_downloadable"]:
        return True

    # --------------------------------------------------------
    # Age restriction
    # --------------------------------------------------------

    if obj["is_age_restricted"]:
        return True

    # --------------------------------------------------------
    # Categories
    # --------------------------------------------------------

    for category in obj["categories"]:

        if category in EXCLUDED_CATEGORIES:
            return True

    # --------------------------------------------------------
    # Name
    # --------------------------------------------------------

    for keyword in EXCLUDED_KEYWORDS:

        if keyword in obj["name"]:
            return True

    # --------------------------------------------------------
    # Tags
    # --------------------------------------------------------

    for tag in obj["tags"]:

        if tag in EXCLUDED_KEYWORDS:
            return True

    # --------------------------------------------------------
    # Geometry
    # --------------------------------------------------------

    try:

        faces = int(
            obj["face_count"]
        )

    except:

        return True

    if faces < MIN_FACES:
        return True

    if faces > MAX_FACES:
        return True

    return False

In [15]:
# ============================================================
# OBJECT FILTERING
# ============================================================

MIN_FACES = 500
MAX_FACES = 500_000


def is_excluded(obj):

    # --------------------------------------------------------
    # License
    # --------------------------------------------------------

    normalized_license = normalize_license(
        obj["license_raw"]
    )

    if normalized_license not in ALLOWED_LICENSES:
        return True

    # --------------------------------------------------------
    # Downloadability
    # --------------------------------------------------------

    if not obj["is_downloadable"]:
        return True

    # --------------------------------------------------------
    # Age restriction
    # --------------------------------------------------------

    if obj["is_age_restricted"]:
        return True

    # --------------------------------------------------------
    # Categories
    # --------------------------------------------------------

    for category in obj["categories"]:

        if category in EXCLUDED_CATEGORIES:
            return True

    # --------------------------------------------------------
    # Name
    # --------------------------------------------------------

    for keyword in EXCLUDED_KEYWORDS:

        if keyword in obj["name"]:
            return True

    # --------------------------------------------------------
    # Tags
    # --------------------------------------------------------

    for tag in obj["tags"]:

        if tag in EXCLUDED_KEYWORDS:
            return True

    # --------------------------------------------------------
    # Geometry
    # --------------------------------------------------------

    try:

        faces = int(
            obj["face_count"]
        )

    except:

        return True

    if faces < MIN_FACES:
        return True

    if faces > MAX_FACES:
        return True

    return False

In [17]:
# ============================================================
# CELL 15 — BUILD CANDIDATE POOLS
# ============================================================

print("=" * 70)
print("BUILDING CANDIDATE POOLS")
print("=" * 70)

# Make absolutely sure the required variables exist
required = [
    "annotations",
    "TARGET_CATEGORIES",
    "is_excluded",
    "extract_metadata",
    "classify_object"
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "The following required variables/functions are missing:\n"
        + "\n".join(f" - {x}" for x in missing)
        + "\n\nRun the earlier cells before Cell 15."
    )


# ------------------------------------------------------------
# Candidate pools
# ------------------------------------------------------------

candidate_pool = {
    category: []
    for category in TARGET_CATEGORIES
}

seen_uids = set()

print(
    f"Scanning {len(annotations):,} Objaverse objects..."
)

print(
    f"Target categories: {len(TARGET_CATEGORIES)}"
)

print()


# ------------------------------------------------------------
# Scan annotations
# ------------------------------------------------------------

for uid, metadata in tqdm(
    annotations.items(),
    total=len(annotations),
    desc="Scanning Objaverse"
):

    # --------------------------------------------------------
    # Never process the same UID twice
    # --------------------------------------------------------

    if uid in seen_uids:
        continue


    # --------------------------------------------------------
    # Extract metadata
    # --------------------------------------------------------

    try:

        obj = extract_metadata(
            uid,
            metadata
        )

    except Exception:

        continue


    # --------------------------------------------------------
    # General filtering
    # --------------------------------------------------------

    try:

        if is_excluded(obj):
            continue

    except Exception:

        continue


    # --------------------------------------------------------
    # Determine category
    # --------------------------------------------------------

    try:

        category = classify_object(
            obj
        )

    except Exception:

        continue


    if category is None:
        continue


    # --------------------------------------------------------
    # Keep candidate pool manageable
    #
    # We only need 20 final objects per category.
    # Keeping 100 candidates gives us enough choice.
    # --------------------------------------------------------

    if len(
        candidate_pool[category]
    ) >= 100:

        continue


    # --------------------------------------------------------
    # Add normalized license
    # --------------------------------------------------------

    obj[
        "license_normalized"
    ] = normalize_license(
        obj["license_raw"]
    )

    obj[
        "category"
    ] = category


    # --------------------------------------------------------
    # Store candidate
    # --------------------------------------------------------

    candidate_pool[
        category
    ].append(obj)

    seen_uids.add(uid)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print()
print("=" * 70)
print("CANDIDATE POOL RESULTS")
print("=" * 70)

total_candidates = 0

for category in TARGET_CATEGORIES:

    count = len(
        candidate_pool[category]
    )

    total_candidates += count

    print(
        f"{category:18s} : {count:3d}"
    )

print("-" * 70)

print(
    f"TOTAL CANDIDATES   : {total_candidates}"
)

print("=" * 70)

BUILDING CANDIDATE POOLS
Scanning 798,759 Objaverse objects...
Target categories: 10



Scanning Objaverse:   0%|          | 0/798759 [00:00<?, ?it/s]


CANDIDATE POOL RESULTS
furniture          : 100
kitchen            : 100
electronics        : 100
tools              : 100
lighting           : 100
clothing           : 100
toys               : 100
musical            : 100
sports             : 100
decorative         : 100
----------------------------------------------------------------------
TOTAL CANDIDATES   : 1000


In [18]:
# ============================================================
# VERIFY CANDIDATE POOLS
# ============================================================

insufficient = []

for category, objects in (
    candidate_pool.items()
):

    if len(objects) < OBJECTS_PER_CATEGORY:

        insufficient.append(
            (
                category,
                len(objects)
            )
        )

if insufficient:

    print(
        "WARNING: Some categories "
        "do not have enough candidates."
    )

    for category, count in insufficient:

        print(
            category,
            count
        )

    raise RuntimeError(
        "Not enough candidates."
    )

else:

    print(
        "✓ Every category has enough "
        "candidates."
    )















✓ Every category has enough candidates.


In [19]:
# ============================================================
# SELECT FINAL 200 OBJECTS
# ============================================================

random.seed(
    RANDOM_SEED
)

selected_objects = []

for category, candidates in (
    candidate_pool.items()
):

    # Copy so we don't modify the pool
    candidates = list(
        candidates
    )

    random.shuffle(
        candidates
    )

    chosen = candidates[
        :OBJECTS_PER_CATEGORY
    ]

    selected_objects.extend(
        chosen
    )

print(
    "Total selected:",
    len(selected_objects)
)

Total selected: 200


In [20]:
# ============================================================
# ASSIGN PERMANENT DATASET IDS
# ============================================================

# Shuffle globally while preserving
# reproducibility.

random.seed(
    RANDOM_SEED
)

random.shuffle(
    selected_objects
)

for index, obj in enumerate(
    selected_objects,
    start=1
):

    obj["dataset_id"] = (
        f"OBJ_{index:04d}"
    )

print(
    selected_objects[0]
)

{'uid': '76193ff4b3ab4718a65b06ec8a1d364a', 'name': 'hammer', 'categories': [], 'tags': [], 'license_raw': 'by', 'is_downloadable': True, 'is_age_restricted': False, 'face_count': 46464, 'vertex_count': 23234, 'viewer_url': 'https://sketchfab.com/3d-models/76193ff4b3ab4718a65b06ec8a1d364a', 'source_uri': 'https://api.sketchfab.com/v3/models/76193ff4b3ab4718a65b06ec8a1d364a', 'creator': 'simon.svanholt', 'license_normalized': 'CC-BY', 'category': 'tools', 'dataset_id': 'OBJ_0001'}


In [21]:
# ============================================================
# VERIFY CATEGORY BALANCE
# ============================================================

category_counts = Counter(
    obj["category"]
    for obj in selected_objects
)

print(
    "CATEGORY DISTRIBUTION"
)

print(
    "=" * 50
)

for category in TARGET_CATEGORIES:

    print(
        f"{category:15s}: "
        f"{category_counts[category]}"
    )

print(
    "=" * 50
)

print(
    "TOTAL:",
    sum(
        category_counts.values()
    )
)

CATEGORY DISTRIBUTION
furniture      : 20
kitchen        : 20
electronics    : 20
tools          : 20
lighting       : 20
clothing       : 20
toys           : 20
musical        : 20
sports         : 20
decorative     : 20
TOTAL: 200


In [22]:
# ============================================================
# CREATE SELECTION METADATA
# ============================================================

selection_records = []

for obj in selected_objects:

    selection_records.append({

        "dataset_id":
            obj["dataset_id"],

        "objaverse_uid":
            obj["uid"],

        "name":
            obj["name"],

        "category":
            obj["category"],

        "license_raw":
            obj["license_raw"],

        "license_normalized":
            obj["license_normalized"],

        "face_count":
            obj["face_count"],

        "vertex_count":
            obj["vertex_count"],

        "is_downloadable":
            obj["is_downloadable"],

        "is_age_restricted":
            obj["is_age_restricted"],

        "creator":
            obj["creator"],

        "source_uri":
            obj["source_uri"],

        "viewer_url":
            obj["viewer_url"]
    })

selection_df = pd.DataFrame(
    selection_records
)

selection_file = (
    METADATA_DIR /
    "selection.csv"
)

selection_df.to_csv(
    selection_file,
    index=False
)

print(
    "Saved:",
    selection_file
)

Saved: /content/neural_object_reconstruction/metadata/selection.csv


In [23]:
# ============================================================
# SAVE SELECTION JSON
# ============================================================

selection_json = (
    METADATA_DIR /
    "selection.json"
)

with open(
    selection_json,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        selection_records,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Saved:",
    selection_json
)

Saved: /content/neural_object_reconstruction/metadata/selection.json


In [24]:
# ============================================================
# CREATE MANIFEST
# ============================================================

manifest_df = selection_df.copy()

manifest_df[
    "status"
] = "selected"

manifest_df[
    "local_file"
] = ""

manifest_df[
    "remote_file"
] = ""

manifest_df[
    "error"
] = ""

manifest_file = (
    METADATA_DIR /
    "manifest.csv"
)

manifest_df.to_csv(
    manifest_file,
    index=False
)

print(
    "Manifest created:"
)

print(
    manifest_file
)

Manifest created:
/content/neural_object_reconstruction/metadata/manifest.csv


In [27]:
# ============================================================
# DATASET README
# ============================================================

README = f"""
---
license: other
task_categories:
- computer-vision
- image-to-image
tags:
- 3d-reconstruction
- gaussian-splatting
- neural-rendering
- relighting
- computer-vision
- blender
- objaverse
---

# Neural Object Reconstruction & Relighting Lab
## Raw 3D Object Dataset

This repository contains the curated raw 3D object collection used
for the Neural Object Reconstruction & Relighting Lab.

## Purpose

The objects are used to generate a synthetic multi-view training
dataset for research into:

- neural 3D reconstruction
- depth estimation
- foreground segmentation
- neural material decomposition
- inverse rendering
- relighting
- 3D Gaussian Splatting

## Source

The raw 3D assets were selected from the Objaverse ecosystem.

Each object retains its original:

- Objaverse UID
- source URI
- Sketchfab viewer URL
- creator information where available
- original license metadata

## Dataset size

200 objects.

## Categories

- Furniture
- Kitchen / Dining
- Electronics
- Tools
- Lighting
- Clothing / Accessories
- Toys / Games
- Musical Instruments
- Sports / Outdoor
- Decorative / Household

20 objects are selected from each category.

## Important licensing note

The license information for each asset is preserved in
`metadata/selection.csv`.

`license_raw` records the original metadata value.

`license_normalized` records the normalized interpretation used
by this pipeline.

Users should independently verify the applicable license and
attribution requirements before redistributing or commercially
using individual assets.

## Dataset structure
metadata/
selection.csv
selection.json
manifest.csv

objects/
OBJ_0001.glb
OBJ_0002.glb
...
OBJ_0200.glb

## Pipeline

Objaverse
→ curated object selection
→ Blender procedural scene generation
→ multi-view rendering
→ RGB
→ foreground masks
→ depth
→ normals
→ albedo
→ roughness
→ specular
→ illumination
→ training dataset

## Research project

Neural Object Reconstruction & Relighting Lab

The final research system combines segmentation,
depth estimation, 3D Gaussian reconstruction,
neural material decomposition and real-time WebGPU rendering.
"""

README_FILE = (
    PROJECT_DIR /
    "README.md"
)

README_FILE.write_text(
    README,
    encoding="utf-8"
)

print(
    "README created."
)

README created.


In [28]:
# ============================================================
# UPLOAD INITIAL METADATA
# ============================================================

METADATA_UPLOAD_DIR = (
    PROJECT_DIR /
    "initial_metadata"
)

METADATA_UPLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True
)

shutil.copy2(
    README_FILE,
    METADATA_UPLOAD_DIR /
    "README.md"
)

shutil.copy2(
    selection_file,
    METADATA_UPLOAD_DIR /
    "selection.csv"
)

shutil.copy2(
    selection_json,
    METADATA_UPLOAD_DIR /
    "selection.json"
)

shutil.copy2(
    manifest_file,
    METADATA_UPLOAD_DIR /
    "manifest.csv"
)

api.upload_folder(
    folder_path=str(
        METADATA_UPLOAD_DIR
    ),
    path_in_repo="",
    repo_id=HF_DATASET,
    repo_type="dataset",
    commit_message=(
        "Initialize dataset metadata"
    )
)

print(
    "✓ Metadata uploaded."
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:11610: UserWarning: Warnings while validating metadata in README.md:
- The task_categories "computer-vision" is not in the official list: text-classification, token-classification, table-question-answering, question-answering, zero-shot-classification, translation, summarization, feature-extraction, text-generation, fill-mask, sentence-similarity, text-to-speech, text-to-audio, automatic-speech-recognition, audio-to-audio, audio-classification, audio-text-to-text, voice-activity-detection, depth-estimation, image-classification, object-detection, image-segmentation, text-to-image, image-to-text, image-to-image, image-to-video, unconditional-image-generation, video-classification, reinforcement-learning, robotics, tabular-classification, tabular-regression, tabular-to-text, table-to-text, multiple-choice, text-ranking, text-retrieval, time-series-forecasting, text-to-video, image-text-to-text, image-text-to-image, image-t

✓ Metadata uploaded.


In [29]:
# ============================================================
# OBJAVERSE DOWNLOAD FUNCTION
# ============================================================

def download_object_to_batch(
    obj,
    batch_dir
):

    dataset_id = obj[
        "dataset_id"
    ]

    uid = obj[
        "uid"
    ]

    destination = (
        batch_dir /
        f"{dataset_id}.glb"
    )

    # If already downloaded in this batch
    if destination.exists():

        print(
            f"✓ {dataset_id} already exists"
        )

        return destination

    try:

        objects = objaverse.load_objects(
            uids=[uid]
        )

        if uid not in objects:

            raise RuntimeError(
                "Objaverse did not return "
                f"object {uid}"
            )

        source = Path(
            objects[uid]
        )

        if not source.exists():

            raise RuntimeError(
                f"Downloaded file does not exist: "
                f"{source}"
            )

        shutil.copy2(
            source,
            destination
        )

        return destination

    except Exception:

        raise

In [30]:
# ============================================================
# BATCH MANAGEMENT
# ============================================================

def create_batches(
    objects,
    batch_size
):

    return [
        objects[i:i + batch_size]

        for i in range(
            0,
            len(objects),
            batch_size
        )
    ]


batches = create_batches(
    selected_objects,
    BATCH_SIZE
)

print(
    "Total batches:",
    len(batches)
)

for i, batch in enumerate(
    batches,
    start=1
):

    print(
        f"Batch {i:02d}: "
        f"{len(batch)} objects"
    )

Total batches: 8
Batch 01: 25 objects
Batch 02: 25 objects
Batch 03: 25 objects
Batch 04: 25 objects
Batch 05: 25 objects
Batch 06: 25 objects
Batch 07: 25 objects
Batch 08: 25 objects


In [31]:
# ============================================================
# FIND OBJECTS ALREADY ON HUGGING FACE
# ============================================================

def get_remote_object_ids():

    files = api.list_repo_files(
        repo_id=HF_DATASET,
        repo_type="dataset"
    )

    remote_ids = set()

    for file in files:

        if not file.startswith(
            "objects/"
        ):
            continue

        filename = Path(
            file
        ).name

        if filename.startswith(
            "OBJ_"
        ):

            remote_ids.add(
                Path(filename).stem
            )

    return remote_ids


remote_ids = (
    get_remote_object_ids()
)

print(
    "Objects already on HF:",
    len(remote_ids)
)

Objects already on HF: 0


In [32]:
# ============================================================
# DOWNLOAD + UPLOAD BATCH
# ============================================================

def process_batch(
    batch,
    batch_number
):

    print()
    print("=" * 80)

    print(
        f"PROCESSING BATCH "
        f"{batch_number:03d}"
    )

    print(
        f"Objects in batch: "
        f"{len(batch)}"
    )

    print("=" * 80)

    batch_dir = (
        BATCHES_DIR /
        f"batch_{batch_number:03d}"
    )

    batch_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Check remote state
    # --------------------------------------------------------

    remote_ids = (
        get_remote_object_ids()
    )

    already_uploaded = [
        obj
        for obj in batch
        if obj["dataset_id"]
        in remote_ids
    ]

    pending = [
        obj
        for obj in batch
        if obj["dataset_id"]
        not in remote_ids
    ]

    print(
        f"Already on HF: "
        f"{len(already_uploaded)}"
    )

    print(
        f"Need upload: "
        f"{len(pending)}"
    )

    # --------------------------------------------------------
    # Download pending objects
    # --------------------------------------------------------

    successful_downloads = []

    for obj in tqdm(
        pending,
        desc=f"Downloading batch {batch_number:03d}"
    ):

        try:

            path = (
                download_object_to_batch(
                    obj,
                    batch_dir
                )
            )

            successful_downloads.append(
                obj
            )

        except Exception as e:

            print()
            print(
                f"✗ {obj['dataset_id']} "
                f"FAILED"
            )

            print(
                "UID:",
                obj["uid"]
            )

            print(
                "Error:",
                e
            )

            manifest_df.loc[
                manifest_df[
                    "dataset_id"
                ] == obj["dataset_id"],
                "status"
            ] = "download_failed"

            manifest_df.loc[
                manifest_df[
                    "dataset_id"
                ] == obj["dataset_id"],
                "error"
            ] = str(e)

    # --------------------------------------------------------
    # If there is nothing to upload
    # --------------------------------------------------------

    files_to_upload = list(
        batch_dir.glob("*.glb")
    )

    if not files_to_upload:

        print(
            "\nNo new files to upload."
        )

    else:

        print()
        print(
            f"Uploading {len(files_to_upload)} "
            f"objects as ONE folder operation..."
        )

        # ----------------------------------------------------
        # ONE upload_folder operation
        # ----------------------------------------------------

        api.upload_folder(
            folder_path=str(
                batch_dir
            ),
            path_in_repo="objects",
            repo_id=HF_DATASET,
            repo_type="dataset",
            commit_message=(
                f"Upload object batch "
                f"{batch_number:03d}"
            )
        )

        print(
            "\n✓ Batch upload finished."
        )

    # --------------------------------------------------------
    # Update local manifest
    # --------------------------------------------------------

    for obj in batch:

        dataset_id = obj[
            "dataset_id"
        ]

        if dataset_id in remote_ids:

            status = "complete"

        elif any(
            x["dataset_id"] == dataset_id
            for x in successful_downloads
        ):

            status = "uploaded"

        else:

            continue

        manifest_df.loc[
            manifest_df[
                "dataset_id"
            ] == dataset_id,
            "status"
        ] = status

        manifest_df.loc[
            manifest_df[
                "dataset_id"
            ] == dataset_id,
            "remote_file"
        ] = (
            f"objects/"
            f"{dataset_id}.glb"
        )

    # --------------------------------------------------------
    # Save local manifest
    # --------------------------------------------------------

    manifest_df.to_csv(
        manifest_file,
        index=False
    )

    print(
        "\nManifest saved locally."
    )

    # --------------------------------------------------------
    # Remove local batch
    # --------------------------------------------------------

    print(
        "\nRemoving local batch..."
    )

    shutil.rmtree(
        batch_dir,
        ignore_errors=True
    )

    gc.collect()

    print(
        "✓ Local batch removed."
    )

    print(
        f"✓ BATCH {batch_number:03d} COMPLETE"
    )

In [33]:
# ============================================================
# RUN COMPLETE DATASET PIPELINE
# ============================================================

for batch_number, batch in enumerate(
    batches,
    start=1
):

    process_batch(
        batch=batch,
        batch_number=batch_number
    )

    print()
    print(
        "#" * 80
    )

    print(
        f"Finished batch "
        f"{batch_number} / "
        f"{len(batches)}"
    )

    print(
        "#" * 80
    )


PROCESSING BATCH 001
Objects in batch: 25
Already on HF: 0
Need upload: 25


Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects

Uploading 25 objects as ONE folder operation...

✓ Batch upload finished.

Manifest saved locally.

Removing local batch...
✓ Local batch removed.
✓ BATCH 001 COMPLETE

################################################################################
Finished batch 1 / 8
################################################################################

PROCESSING BATCH 002


Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects

Uploading 25 objects as ONE folder operation...

✓ Batch upload finished.

Manifest saved locally.

Removing local batch...
✓ Local batch removed.
✓ BATCH 002 COMPLETE

################################################################################
Finished batch 2 / 8
################################################################################

PROCESSING BATCH 003


Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects

Uploading 25 objects as ONE folder operation...

✓ Batch upload finished.

Manifest saved locally.

Removing local batch...
✓ Local batch removed.
✓ BATCH 003 COMPLETE

################################################################################
Finished batch 3 / 8
################################################################################

PROCESSING BATCH 004
Objects in batch: 25
Alre

Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects

Uploading 25 objects as ONE folder operation...

✓ Batch upload finished.

Manifest saved locally.

Removing local batch...
✓ Local batch removed.
✓ BATCH 004 COMPLETE

################################################################################
Finished batch 4 / 8
################################################################################

PROCESSING BATCH 005
Objects in batch: 25
Alre

Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects

Uploading 25 objects as ONE folder operation...

✓ Batch upload finished.

Manifest saved locally.

Removing local batch...
✓ Local batch removed.
✓ BATCH 005 COMPLETE

################################################################################
Finished batch 5 / 8
################################################################################

PROCESSING BATCH 006


Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects

Uploading 25 objects as ONE folder operation...

✓ Batch upload finished.

Manifest saved locally.

Removing local batch...
✓ Local batch removed.
✓ BATCH 006 COMPLETE

################################################################################
Finished batch 6 / 8
################################################################################

PROCESSING BATCH 007
Objects in batch: 25
Alre

Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects

Uploading 25 objects as ONE folder operation...

✓ Batch upload finished.

Manifest saved locally.

Removing local batch...
✓ Local batch removed.
✓ BATCH 007 COMPLETE

################################################################################
Finished batch 7 / 8
################################################################################

PROCESSING BATCH 008
Objects in batch: 25
Already on HF: 0
Need upload:

Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects
Downloaded 1 / 1 objects

Uploading 25 objects as ONE folder operation...

✓ Batch upload finished.

Manifest saved locally.

Removing local batch...
✓ Local batch removed.
✓ BATCH 008 COMPLETE

################################################################################
Finished batch 8 / 8
################################################################################


In [34]:
# ============================================================
# FINAL DATASET VERIFICATION
# ============================================================

remote_ids = (
    get_remote_object_ids()
)

print(
    "Objects on Hugging Face:",
    len(remote_ids)
)

expected_ids = {
    obj["dataset_id"]
    for obj in selected_objects
}

missing = (
    expected_ids -
    remote_ids
)

unexpected = (
    remote_ids -
    expected_ids
)

print(
    "\nExpected:",
    len(expected_ids)
)

print(
    "Uploaded:",
    len(remote_ids)
)

print(
    "Missing:",
    len(missing)
)

print(
    "Unexpected:",
    len(unexpected)
)

if missing:

    print(
        "\nMISSING OBJECTS:"
    )

    for x in sorted(missing):

        print(x)

else:

    print(
        "\n✓ ALL 200 OBJECTS ARE PRESENT"
    )

Objects on Hugging Face: 200

Expected: 200
Uploaded: 200
Missing: 0
Unexpected: 0

✓ ALL 200 OBJECTS ARE PRESENT


In [35]:
# ============================================================
# FINAL MANIFEST UPDATE
# ============================================================

for obj in selected_objects:

    dataset_id = obj[
        "dataset_id"
    ]

    if dataset_id in remote_ids:

        manifest_df.loc[
            manifest_df[
                "dataset_id"
            ] == dataset_id,
            "status"
        ] = "complete"

        manifest_df.loc[
            manifest_df[
                "dataset_id"
            ] == dataset_id,
            "remote_file"
        ] = (
            f"objects/"
            f"{dataset_id}.glb"
        )

manifest_df.to_csv(
    manifest_file,
    index=False
)

api.upload_file(
    path_or_fileobj=str(
        manifest_file
    ),
    path_in_repo=(
        "metadata/manifest.csv"
    ),
    repo_id=HF_DATASET,
    repo_type="dataset",
    commit_message=(
        "Finalize object manifest"
    )
)

print(
    "✓ Final manifest uploaded."
)

✓ Final manifest uploaded.


In [36]:
# ============================================================
# FINAL REPOSITORY CONTENTS
# ============================================================

files = api.list_repo_files(
    repo_id=HF_DATASET,
    repo_type="dataset"
)

print(
    "Repository contents:"
)

for file in files:

    print(file)

Repository contents:
.gitattributes
README.md
manifest.csv
metadata/manifest.csv
objects/OBJ_0001.glb
objects/OBJ_0002.glb
objects/OBJ_0003.glb
objects/OBJ_0004.glb
objects/OBJ_0005.glb
objects/OBJ_0006.glb
objects/OBJ_0007.glb
objects/OBJ_0008.glb
objects/OBJ_0009.glb
objects/OBJ_0010.glb
objects/OBJ_0011.glb
objects/OBJ_0012.glb
objects/OBJ_0013.glb
objects/OBJ_0014.glb
objects/OBJ_0015.glb
objects/OBJ_0016.glb
objects/OBJ_0017.glb
objects/OBJ_0018.glb
objects/OBJ_0019.glb
objects/OBJ_0020.glb
objects/OBJ_0021.glb
objects/OBJ_0022.glb
objects/OBJ_0023.glb
objects/OBJ_0024.glb
objects/OBJ_0025.glb
objects/OBJ_0026.glb
objects/OBJ_0027.glb
objects/OBJ_0028.glb
objects/OBJ_0029.glb
objects/OBJ_0030.glb
objects/OBJ_0031.glb
objects/OBJ_0032.glb
objects/OBJ_0033.glb
objects/OBJ_0034.glb
objects/OBJ_0035.glb
objects/OBJ_0036.glb
objects/OBJ_0037.glb
objects/OBJ_0038.glb
objects/OBJ_0039.glb
objects/OBJ_0040.glb
objects/OBJ_0041.glb
objects/OBJ_0042.glb
objects/OBJ_0043.glb
objects/OBJ_0044

In [37]:
# ============================================================
# FINAL RESULT
# ============================================================

print("=" * 80)

print(
    "DATASET COMPLETE"
)

print("=" * 80)

print()

print(
    "Hugging Face dataset:"
)

print(
    f"https://huggingface.co/datasets/{HF_DATASET}"
)

print()

print(
    "Objects:",
    len(remote_ids)
)

print(
    "Categories:",
    len(TARGET_CATEGORIES)
)

print(
    "Objects per category:",
    OBJECTS_PER_CATEGORY
)

print()

print(
    "Status:",
    "READY FOR BLENDER"
    if len(remote_ids) == TOTAL_OBJECTS
    else "INCOMPLETE"
)

DATASET COMPLETE

Hugging Face dataset:
https://huggingface.co/datasets/ndeda/neural-object-reconstruction-raw

Objects: 200
Categories: 10
Objects per category: 20

Status: READY FOR BLENDER
